# Guía comentada: cómo leer este estudio desde cero

**Para quién es este cuaderno.** Para alguien que nunca ha trabajado con precios, series de tiempo
ni modelos predictivos. No supone conocimientos de economía ni de estadística. Cada concepto se
explica con una analogía, se muestra en una tabla o figura, y se interpreta.

**Qué vamos a entender.** Si el precio de un alimento sube en Lima, ¿también sube en Puno? ¿Cuánto
tarda en llegar ese aumento? ¿Se puede anticipar el precio de un departamento mirando el de Lima?

**Cómo está organizado.** Cada sección tiene tres partes:

1. **Qué se hace** (la idea, sin tecnicismos).
2. **Cómo se ve** (la tabla o figura).
3. **Cómo se lee** (qué mirar y qué conclusión se saca).

Al final hay un glosario y un índice de artefactos.

## 0. El mapa del estudio (flujo KDD)

Antes de entrar en los datos, conviene ver el recorrido completo. La figura muestra las cinco
fases del proceso KDD (*Knowledge Discovery in Databases*): selección, preprocesamiento,
transformación, minería y evaluación. Cada fase produce algo que la siguiente necesita.

![](../outputs/informe/figuras/01_flujo_kdd.png)

### Detalle elemento por elemento: figura 01

**Qué muestra.** Las cinco fases del proceso KDD, en orden.

**Los elementos.**
- **Cinco cajas:** selección, preprocesamiento, transformación, minería y evaluación.
- **Flechas:** indican que cada fase alimenta a la siguiente. No se puede minar sin transformar, ni
  transformar sin limpiar.
- **Texto pequeño dentro de cada caja:** lo que se hizo en este estudio en esa fase.

**Qué hay en cada caja.**
1. **Selección:** se eligió el SISAP, 24 departamentos y 9 productos.
2. **Preprocesamiento:** se auditaron nulos, duplicados, unidades y atípicos.
3. **Transformación:** se normalizó a S/ kg, se aplicó log, se crearon cambios, rezagos y variables
   estacionales.
4. **Minería:** se estimó el ECM y se compararon cuatro modelos predictivos.
5. **Evaluación:** se usaron HAC, bootstrap, R², MAE, mejora porcentual frente a la caminata aleatoria,
   acierto direccional y Diebold–Mariano.

**Cómo se lee.** El orden no es decorativo: si se invierte, se modela sobre datos sucios o se
evalúa con una métrica que no corresponde. La figura es el mapa del resto del cuaderno.

## 1. La idea del estudio en una frase

Lima es el mercado mayorista más grande del país. Si los mercados están **conectados**, un cambio
de precio en Lima debería aparecer, con algún retraso, en los demás departamentos. Si están **aislados**,
no. El estudio mide cuánto se transmite, en cuánto tiempo y si esa relación sirve para anticipar.

**Analogía.** Imagina una red de cañerías: Lima es el tanque principal y cada departamento es un grifo.
Si abres el tanque, el agua llega a todos los grifos, pero a unos rápido y a otros lento, y a
algunos casi no llega. La pregunta es: ¿qué tan abiertas están las cañerías entre Lima y cada
departamento?

**Lo que NO es el estudio.** No dice si el precio es “justo”, no mide la inflación oficial y no
predice el futuro con una bola de cristal. Mide una relación histórica y su utilidad predictiva,
con límites declarados.

## 2. Glosario mínimo

| Término | Qué significa en este estudio |
|---|---|
| **Precio mayorista** | Precio al que se vende por volumen en un mercado grande, no el que paga el consumidor final en la bodega. |
| **S/** | Soles peruanos, la moneda. |
| **kg** | Kilogramo. Algunos mercados venden por arroba o saco; todo se convierte a kilos para comparar. |
| **Departamento** | Unidad transversal oficial del estudio (Puno, Cusco, Lima…). |
| **Categoría** | Uno de los 9 productos armonizados del panel (papa, cebolla, tomate, etc.). |
| **Variedad** | Presentación comercial del producto en la fuente, como papa canchan o papa amarilla. No equivale a una categoría adicional en el panel. |
| **Mes** | La unidad de tiempo. Cada fila es una categoría en un departamento en un mes. |
| **Serie de tiempo** | Una variable medida repetidamente en el tiempo (por ejemplo, el precio de la categoría papa en Puno mes a mes). |
| **Logaritmo** | Operación que convierte razones multiplicativas en diferencias; reduce la asimetría observada, pero no garantiza varianza constante. |
| **Retorno o cambio** | Diferencia de log-precios; aproxima la variación porcentual para cambios pequeños. |
| **Correlación** | Número entre −1 y 1 que mide si dos series se mueven juntas. |
| **Estacionariedad** | Propiedad de una serie cuyo nivel medio y dispersión no cambian sistemáticamente con el tiempo. |
| **Modelo** | Regla matemática que relaciona variables; se estima con datos. |
| **MAE** | Error absoluto medio: cuánto se equivoca un pronóstico, en promedio. |
| **R²** | Proporción de la variación que el modelo logra explicar. |
| **Nowcast** | Estimar el valor del mes actual con información de ese mes. |
| **Pronóstico** | Estimar el valor del mes siguiente. |

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from IPython.display import display, Markdown

RAIZ = Path.cwd()
if not (RAIZ / "data").exists():
    RAIZ = RAIZ.parent
TAB = RAIZ / "outputs" / "informe" / "tablas"
FIG = RAIZ / "outputs" / "informe" / "figuras"

panel = pd.read_csv(RAIZ / "data" / "processed" / "panel_precios_limpio.csv", parse_dates=["fecha"])
print(f"El panel tiene {len(panel):,} filas.")
print(f"Hay {panel.region.nunique()} departamentos y {panel.categoria.nunique()} productos.")
print(f"Va de {panel.fecha.min():%Y-%m} a {panel.fecha.max():%Y-%m}.")

El panel tiene 43,855 filas.
Hay 24 departamentos y 9 productos.
Va de 1997-01 a 2026-11.


## 3. Qué contiene una fila

Cada fila del panel analítico es una observación de **una categoría, en un departamento y en un mes**.
`precio_kg` indica cuánto costaba un kilogramo y `n_registros` cuántos precios se promediaron.

La fuente tiene variedades comerciales: 532 series departamento-variedad se armonizaron en 216
series analíticas departamento-categoría (24 × 9). Por eso una variedad alternativa como papa
amarilla no se cuenta como una décima categoría del panel principal. Se excluyen los mercados
provinciales Andahuaylas, Chota y Jaén, y Callao, para mantener 24 departamentos. El catálogo
`data/reference/departamentos.csv` es la fuente única de nombres, códigos y coordenadas.

Veamos las primeras filas reales:

In [2]:
display(panel.head(8))

,fecha,anio,mes,departamento_id,departamento,region,categoria,n_registros,precio_kg
0,2008-09-01,2008,9,PE-AMA,Amazonas,Amazonas,Aceite,1,5.56
1,2008-10-01,2008,10,PE-AMA,Amazonas,Amazonas,Aceite,1,6.39
2,2008-11-01,2008,11,PE-AMA,Amazonas,Amazonas,Aceite,1,6.72
3,2008-12-01,2008,12,PE-AMA,Amazonas,Amazonas,Aceite,1,6.64
4,2009-01-01,2009,1,PE-AMA,Amazonas,Amazonas,Aceite,1,6.34
5,2009-02-01,2009,2,PE-AMA,Amazonas,Amazonas,Aceite,1,5.71
6,2009-03-01,2009,3,PE-AMA,Amazonas,Amazonas,Aceite,1,5.54
7,2009-04-01,2009,4,PE-AMA,Amazonas,Amazonas,Aceite,1,5.66


**Cómo se lee.** La primera fila, por ejemplo, dice: en tal mes, en tal departamento, el kilogramo de
tal producto costaba tantos soles. No hay magia: es una tabla ordenada.

La tabla de variables del informe resume tipo, unidad, rol y fuente:

In [3]:
display(pd.read_csv(TAB / "tabla_01_variables.csv"))

,variable,tipo,unidad,rol,fuente
0,fecha,fecha,mes,índice temporal,SISAP
1,anio,entero,año,derivada,fecha
2,mes,entero,1-12,derivada,fecha
3,region,texto,24 departamentos,unidad transversal,SISAP
4,categoria,texto,9 categorías,unidad de producto armonizado,SISAP
5,precio_kg,decimal,S/ por kg,variable de análisis,SISAP normalizado
6,n_registros,entero,conteo,control de agregación,derivada


### Cómo se lee la tabla de variables

- **variable:** nombre de la columna en el dataset.
- **tipo:** fecha, entero, texto o decimal.
- **unidad:** mes, año, S/ kg, categoría o conteo.
- **rol:** para qué se usa (índice temporal, unidad transversal o variable de análisis).
- **fuente:** de dónde sale (SISAP o derivada).

La segunda tabla muestra las primeras filas reales. Cada fila corresponde a una categoría armonizada
en un departamento y un mes.

## 4. Cobertura: no todos los meses existen para todos los departamentos

El portal ofrece datos desde 1997, pero **1997 no significa 24 departamentos**. En el panel analítico,
ese año solo hay Lima y dos categorías. La cobertura departamental se vuelve más amplia en 2002 y densa
en 2012. Por eso el estudio declara una **ventana efectiva** para las estimaciones: 2012–2023.

**Analogía.** Es como un cuaderno de asistencia: el cuaderno existe desde 1997, pero al principio
solo firmaba una persona. No puedes decir que tienes 24 alumnos desde el primer día.

In [4]:
display(pd.read_csv(TAB / "tabla_02_auditoria.csv"))

,metrica,valor
0,filas,43921
1,departamentos,24
2,productos,9
3,nulos,0
4,duplicados fecha-departamento-producto,0
5,atipicos detectados,66
6,atipicos (%),0.15
7,primer mes,1997-01
8,ultimo mes,2026-11
9,cobertura 2012-2023 (%),85.9


### Cómo se lee la auditoría

- **filas:** total de observaciones del panel.
- **departamentos / productos:** cuántos hay.
- **nulos:** celdas vacías. Aquí son 0 porque el panel se construyó con precios completos.
- **duplicados:** filas repetidas para la misma fecha, departamento y producto. Aquí son 0.
- **atípicos detectados:** valores incompatibles con el entorno del precio (66).
- **cobertura:** porcentaje de la grilla teórica que está lleno.

La tabla de verificación de limpieza compara antes y después: cuántas filas se descartaron, cuántos
nulos y duplicados quedan, y cómo cambia la cobertura.

![](../outputs/informe/figuras/02_cobertura.png)

### Detalle elemento por elemento: figura 02

**Qué muestra.** Cuántas departamentos y categorías del **panel armonizado** tienen datos en cada año.
No es el conteo bruto de todos los rótulos de variedad en la fuente: esta figura usa las 9
categorías analizadas. Por eso, en 2000 marca **22 de los 24 departamentos y 7 categorías**, igual que la Tabla 2;
la auditoría de fuente cruda cuenta una capa distinta.

**Ejes.**
- **Eje X (horizontal):** el año, de 1997 a 2026.
- **Eje Y (vertical):** cantidad de departamentos o categorías con datos ese año.

**Las dos líneas.**
- **Línea azul con círculos:** número de departamentos con al menos un precio en el panel final.
- **Línea naranja con cuadrados:** número de categorías del panel con datos.

**Cómo se lee.** En 2000 hay cobertura fragmentaria; desde 2002 el panel regional se amplía y a
partir de 2012 se vuelve denso. Una línea alta no significa que todas las combinaciones departamento-
categoría tengan datos.

**Qué conclusión saca.** La serie nominal empieza en 1997, pero el panel regional comparable
empieza después y la ventana principal de estimación es 2012–2023.

**Cómo se lee la figura.** La línea de departamentos y la de categorías aumentan en momentos distintos.
La meseta indica cuándo el panel se vuelve más comparable; aun entonces, algunas celdas tienen
meses faltantes.

## 5. Limpieza: por qué hay que convertir arrobas, sacos y cajas

Un mismo producto puede venderse por kilogramo en Lima, por arroba en Puno y por saco en otro
mercado. Si no se convierte todo a kilos, se comparan unidades distintas. La regla fue:

1. Elegir la presentación dominante de cada celda departamento-producto.
2. Dividir el precio por su equivalencia en kilogramos.

También se detectaron **errores de punto decimal**: un precio de 145,34 S/kg de cebolla cuando la
mediana nacional era 2,36. Son 66 casos (0,15 %). No se inventan valores: se marcan como faltantes.

**Analogía.** Si alguien anota “1,45” como “145”, el promedio se dispara. La limpieza detecta esos
dedazos y los retira.

![](../outputs/informe/figuras/23_verificacion_limpieza.png)

### Detalle elemento por elemento: figura 23

**Qué muestra.** El efecto de la limpieza sobre la distribución del precio de la cebolla.

**Dos paneles.**
- **Izquierda (“antes”):** todos los datos, incluidos los errores de punto decimal.
- **Derecha (“después”):** los mismos datos sin los 66 valores descartados.

**Ejes.**
- **Eje X:** precio en soles por kilogramo. Está recortado en el percentil 99,5 para que el
  gráfico sea legible; si no, un valor de 145 S/kg aplastaría todo lo demás.
- **Eje Y:** frecuencia, es decir, cuántos meses caen en cada rango de precio.

**Las barras.**
- Cada barra es un intervalo de precio; su altura es cuántas observaciones hay en ese intervalo.
- **Antes:** hay una cola larga a la derecha, con valores aislados muy altos.
- **Después:** la cola desaparece; la masa principal queda igual.

**Cómo se lee.** La limpieza no cambia el precio típico: elimina valores imposibles. Si el
gráfico “después” perdiera la mitad de las barras, habría un problema; aquí solo se corta la cola.

In [5]:
display(pd.read_csv(TAB / "tabla_22_verificacion_limpieza.csv"))

,metrica,valor
0,filas antes,43921.0
1,filas despues,43855.0
2,filas descartadas,66.0
3,atipicos marcados,66.0
4,nulos antes,0.0
5,nulos despues,0.0
6,duplicados antes,0.0
7,duplicados despues,0.0
8,cobertura 2012-2023 antes (%),85.9
9,cobertura 2012-2023 despues (%),85.7


### Cómo se lee la verificación de limpieza

- **filas antes / después:** tamaño del panel antes y después de descartar atípicos.
- **filas descartadas:** cuántas se quitaron.
- **atípicos marcados:** cuántos valores se detectaron.
- **nulos / duplicados:** control de calidad; deben ser 0.
- **cobertura antes / después:** porcentaje de la grilla teórica.

Sirve para demostrar que la limpieza fue quirúrgica: quita errores, no datos válidos.

## 6. Describir los datos: media, mediana, dispersión y forma

Antes de modelar, hay que saber cómo se distribuyen los precios.

| Medida | Qué responde |
|---|---|
| **Media** | Valor promedio. Se afecta por valores extremos. |
| **Mediana** | Valor del medio: la mitad está por debajo, la mitad por arriba. Es más robusta. |
| **Desviación estándar** | Cuánto se alejan los valores del promedio. |
| **Percentiles** | Cortes: el p25 deja 25 % abajo; el p75, 75 % abajo. |
| **Asimetría** | Si la cola es más larga a la derecha (positiva) o a la izquierda (negativa). |
| **Curtosis** | Si hay más valores extremos que en una campana normal. |

Veamos un ejemplo con la cebolla:

In [6]:
cebolla = panel.loc[panel.categoria == "Cebolla", "precio_kg"].dropna()
print(f"n = {len(cebolla):,}")
print(f"media   = {cebolla.mean():.2f} S/kg")
print(f"mediana = {cebolla.median():.2f} S/kg")
print(f"desv.   = {cebolla.std():.2f} S/kg")
print(f"p25     = {cebolla.quantile(.25):.2f}")
print(f"p75     = {cebolla.quantile(.75):.2f}")
print(f"mínimo  = {cebolla.min():.2f}")
print(f"máximo  = {cebolla.max():.2f}")
print(f"asimetría = {cebolla.skew():.2f}")

n = 6,135
media   = 1.35 S/kg
mediana = 1.20 S/kg
desv.   = 0.72 S/kg
p25     = 0.83
p75     = 1.69
mínimo  = 0.14
máximo  = 6.67
asimetría = 1.78


**Cómo se lee.** Si la media es mayor que la mediana, hay valores altos que empujan el promedio
hacia arriba: la distribución tiene cola a la derecha. Eso es típico de precios de alimentos.

La tabla completa por producto:

In [7]:
display(pd.read_csv(TAB / "tabla_03_descriptivos.csv").round(3))

,producto,n,media,desv,min,p25,mediana,p75,max,asimetria,curtosis
0,Papa,4074,1.235,0.508,0.370,0.846,1.120,1.532,4.670,1.077,1.719
1,Cebolla,6135,1.346,0.723,0.140,0.826,1.200,1.690,6.670,1.775,5.926
2,Zanahoria,5484,1.277,0.724,0.220,0.800,1.110,1.536,6.930,2.007,6.199
3,Azucar,4795,2.459,0.667,1.051,1.952,2.369,2.812,4.369,0.534,-0.438
4,Arroz,4651,2.686,0.567,1.209,2.303,2.607,3.050,5.400,0.398,0.076
5,Limon,4277,2.241,1.511,0.270,1.295,1.980,2.710,13.500,2.887,13.073
6,Tomate,4314,1.783,0.793,0.430,1.242,1.602,2.100,8.500,1.607,4.335
7,Aceite,3938,7.273,1.738,3.458,6.339,6.750,7.333,13.298,1.552,1.618
8,Huevos,6187,4.753,1.594,1.800,3.660,4.400,5.305,11.190,1.153,0.928


### Cómo se lee la tabla de descriptivos

- **n:** cantidad de observaciones del producto.
- **media:** promedio; se afecta por valores extremos.
- **desv:** desviación estándar; dispersión.
- **min / max:** valores mínimo y máximo.
- **p25 / mediana / p75:** percentiles 25, 50 y 75. La mediana divide los datos en dos mitades.
- **asimetría:** positiva si la cola derecha es más larga.
- **curtosis:** alta si hay más valores extremos que en una normal.

La tabla de transformación log compara la asimetría antes y después del log: si el valor absoluto
baja, el log ayudó.

## 7. Campanas de Gauss: qué son y cómo se leen

Una **campana de Gauss** es la forma de distribución “normal”: simétrica, con la mayoría de los
valores cerca del centro y pocos en los extremos. En la figura:

- Las **barras** son el histograma: cuántas observaciones caen en cada rango de precio.
- La **línea naranja** es una estimación suave de la densidad (KDE).
- La **línea negra punteada** es la campana normal con la misma media y desviación.

Si la línea naranja se parece a la negra, los datos son aproximadamente normales. Si está
“estirada” hacia un lado, no.

![](../outputs/informe/figuras/03_campanas_precio.png)

### Detalle elemento por elemento: figura 03

**Qué muestra.** La distribución del precio de cada categoría, en 9 paneles (3 filas × 3 columnas).

**Ejes.**
- **Eje X:** precio en soles por kilogramo (S/kg).
- **Eje Y:** densidad, no frecuencia; el área total de cada distribución es 1.

**Elementos.** Las barras celestes son el histograma, la línea naranja es una densidad suavizada
(KDE), y la línea negra discontinua es una normal de referencia con la misma media y desviación.

**Cómo se lee.** La papa, cebolla y otras categorías muestran colas derechas. En el azúcar se ven
dos concentraciones en histograma/KDE: es una descripción visual de la figura, no el resultado de
una prueba formal de multimodalidad. En limón la cola derecha es larga.

El eje X se recorta en el percentil 99,5 para que un valor extremo no aplaste el gráfico; los
valores recortados siguen en las tablas y auditoría.

**Cómo se lee.** En papa y cebolla hay cola a la derecha: algunos meses tienen precios mucho más
altos. En azúcar, la figura sugiere dos concentraciones, pero el estudio no aplica una prueba formal
para confirmar bimodalidad. En limón la cola es muy larga.

Por eso se trabaja con logaritmos: comprimen la escala alta y hacen que cambios proporcionales se
expresen como diferencias.

## 8. El logaritmo: por qué se usa

El logaritmo ayuda a expresar y comparar **cambios relativos**. Para precios positivos de dos meses
consecutivos:

`Δlog(P) = log(P_t) − log(P_{t−1}) = log(P_t / P_{t−1})`.

La diferencia logarítmica no es exactamente el porcentaje de cambio. Si `d = Δlog(P)`, el cambio
porcentual exacto es `100 × (exp(d) − 1)`. Cuando el cambio es pequeño, se aproxima con `100 × d`.
Por ejemplo, de 100 a 105 el cambio exacto es 5 %; `d = log(1,05) = 0,0488`, que aproximado
corresponde a 4,88 %. Para un mismo aumento proporcional, la diferencia logarítmica es igual aunque
los precios iniciales sean distintos (por ejemplo, 1 a 1,10 y 10 a 11 son ambos +10 %).

Además, el log comprime las colas altas: en estos datos reduce la asimetría en **9 de 9 productos**.
Eso es un resultado observado, no una garantía de normalidad ni de varianza constante.

In [8]:
display(pd.read_csv(TAB / "tabla_20_transformacion_log.csv").round(3))

,producto,asimetria_precio,asimetria_log,curtosis_precio,curtosis_log,reduce_asimetria,reduce_curtosis
0,Papa,1.077,0.090,1.719,-0.523,True,True
1,Cebolla,1.775,-0.067,5.926,0.054,True,True
2,Zanahoria,2.007,0.107,6.199,0.234,True,True
3,Azucar,0.534,-0.009,-0.438,-0.549,True,True
4,Arroz,0.398,-0.215,0.076,0.068,True,True
5,Limon,2.887,0.005,13.073,0.464,True,True
6,Tomate,1.607,0.159,4.335,0.164,True,True
7,Aceite,1.552,1.102,1.618,0.862,True,True
8,Huevos,1.153,0.413,0.928,-0.158,True,True


### Cómo se lee la tabla de transformación log

- **producto:** producto.
- **asimetria_precio / asimetria_log:** asimetría antes y después del log. Más cerca de 0 = más
  simétrica.
- **curtosis_precio / curtosis_log:** valores extremos antes y después.
- **reduce_asimetria:** verdadero si el valor absoluto de la asimetría baja con el log.
- **reduce_curtosis:** verdadero si la curtosis baja.

El log reduce la asimetría en 9 de 9 productos.

![](../outputs/informe/figuras/04_campanas_log.png)

### Detalle elemento por elemento: figura 04

**Qué muestra.** Lo mismo que la figura 03, pero con el **logaritmo** del precio.

**Ejes.**
- **Eje X:** logaritmo del precio en S/kg. Puede ser negativo; eso **no** significa precio
  negativo. Por ejemplo, `log(0,50) = −0,69`: un precio de medio sol tiene log negativo.
- **Eje Y:** densidad.

**Los tres elementos son los mismos.**
- **Barras celestes:** histograma del log-precio.
- **Línea naranja continua:** KDE del log-precio.
- **Línea negra discontinua:** normal con la misma media y desviación que el log-precio.

**Cómo se lee.**
- Compare esta figura con la 03: la línea naranja ahora se parece más a la negra.
- La asimetría se reduce, pero no desaparece. El log es una mejora, no una garantía de normalidad.
- Los valores negativos del eje X son normales en escala logarítmica.

**Por qué se usa el log.** Porque convierte cambios multiplicativos en sumas, reduce la cola
derecha y hace comparables productos caros y baratos.

**Cómo se lee.** Las distribuciones logarítmicas siguen siendo asimétricas, pero menos. La línea
normal se acerca más al centro. Es una mejora, no un milagro.

### 8.1 Gráfico QQ: otra forma de ver si es normal

El gráfico **QQ** compara los datos con una normal teórica. Si los puntos siguen la línea recta,
los datos son normales. Si se curvan en los extremos, hay colas más pesadas.

![](../outputs/informe/figuras/05_qq_log.png)

### Detalle elemento por elemento: figura 05

**Qué muestra.** Un gráfico **QQ** (*quantile-quantile*) para el log-precio de cada producto.

**Ejes.**
- **Eje X:** cuantiles teóricos de una distribución normal.
- **Eje Y:** cuantiles observados del log-precio.

**Los dos elementos.**
- **Puntos azules:** cada punto compara un cuantil observado con el que tendría una normal.
- **Línea roja recta:** la referencia de normalidad perfecta. Si los datos fueran normales, todos
  los puntos caerían sobre esa línea.

**Cómo se lee.**
- Puntos sobre la línea → distribución aproximadamente normal.
- Puntos que se curvan hacia arriba en el extremo derecho → cola derecha más pesada que la normal.
- Puntos que se curvan hacia abajo en el extremo izquierdo → cola izquierda más pesada.
- Desviaciones solo en los extremos → los datos son casi normales en el centro, pero con outliers.

**Qué conclusión saca.** Las colas son más pesadas de lo normal; por eso se usa MAE y no solo
promedios, y por eso se reportan intervalos.

**Cómo se lee.** Los puntos se separan de la recta en los extremos: hay valores más extremos de lo
que predice una normal. Eso justifica usar métricas robustas (MAE) y no confiar solo en promedios.

## 9. Correlación: ¿dos series se mueven juntas?

La **correlación de Pearson** (`r`) mide relación lineal:

- `r = 1`: se mueven exactamente juntas.
- `r = 0`: no hay relación lineal.
- `r = −1`: se mueven en direcciones opuestas.

Se calcula sobre **cambios mensuales**, no sobre niveles. ¿Por qué? Porque dos series que suben
por inflación se correlacionan en nivel aunque no tengan nada que ver. Los cambios eliminan esa
tendencia común.

**Analogía.** Dos personas que crecen toda la infancia tienen altura correlacionada en nivel, pero
eso no significa que una cause la otra. Lo relevante es si crecen más o menos en los mismos meses.

In [9]:
d = panel[panel.categoria == "Cebolla"].pivot_table(
    index="fecha", columns="region", values="precio_kg").loc["2012":"2023"]
# Reponer los meses del calendario: diff() no debe saltar sobre un mes faltante.
d = d.reindex(pd.date_range(d.index.min(), d.index.max(), freq="MS"))
cambios = np.log(d[["Lima", "Ica", "Puno"]]).diff()
print("Correlación de cambios mensuales (pares de meses consecutivos):")
display(cambios.corr().round(2))

Correlación de cambios mensuales (pares de meses consecutivos):


region,Lima,Ica,Puno
region,,,
Lima,1.00,0.77,0.62
Ica,0.77,1.00,0.57
Puno,0.62,0.57,1.00


**Cómo se lee.** Lima e Ica se mueven casi juntas (rojo fuerte). Lima y Puno se mueven menos. Esa
diferencia es el fenómeno de estudio: no todos los mercados están igual de conectados.

![](../outputs/informe/figuras/06_correlacion_papa.png)

### Detalle elemento por elemento: figura 06

**Qué muestra.** Una **matriz de correlación** entre 10 departamentos, para el precio de la papa.

**Ejes.**
- **Filas:** departamentos.
- **Columnas:** las mismas departamentos.
- Cada celda es la correlación entre los **cambios mensuales** de esas dos departamentos (2012–2023).

**Los elementos.**
- **Diagonal:** siempre 1, porque cada departamento consigo misma está perfectamente correlacionada.
- **Números dentro de cada celda:** el valor de correlación, entre −1 y 1.
- **Color rojo:** correlación positiva alta (se mueven juntos).
- **Color blanco:** correlación cercana a 0 (no hay relación lineal).
- **Color azul:** correlación negativa (se mueven en direcciones opuestas).

**Cómo se lee.**
- Un bloque rojo grande indica un grupo de mercados integrados.
- Celdas casi blancas indican departamentos desconectadas.
- Lima–Ica = 0,85: muy integradas.
- Lima–Tumbes = −0,03: prácticamente nada.

**Advertencia.** Correlación no es causalidad. Dos departamentos pueden correlacionarse porque ambas
siguen a un tercer mercado, no porque una cause a la otra.

**Cómo se lee la matriz.** Cada celda es la correlación entre dos departamentos para papa. La diagonal
es 1 (una serie consigo misma). Rojo = positiva, azul = negativa. Lima–Ica = 0,85; Lima–Tumbes
= −0,03.

![](../outputs/informe/figuras/07_correlacion_lima.png)

### Detalle elemento por elemento: figura 07

**Qué muestra.** La correlación de cada producto con Lima, departamento por departamento.

**Ejes.**
- **Filas:** productos (papa, cebolla, zanahoria, etc.).
- **Columnas:** departamentos.
- Cada celda es la correlación entre los cambios mensuales de ese producto en ese departamento y los
  cambios en Lima.

**Los elementos.**
- **Rojo intenso:** correlación alta con Lima (mercado integrado).
- **Azul:** correlación negativa.
- **Blanco o celeste pálido:** correlación cercana a cero.
- **Espacios en blanco:** no hay suficientes meses en común (< 60) para calcularla.

**Cómo se lee.**
- Una fila con muchos rojos indica un producto ampliamente integrado.
- Una columna pálida indica un departamento aislada para casi todos los productos.
- No todas las filas se ven igual: la integración depende del producto, no solo de el departamento.

**Cómo se lee.** Cada fila es un producto y cada columna un departamento. Los colores muestran cuánto
se parece ese departamento a Lima. No es un bloque uniforme: hay departamentos integradas y departamentos aisladas.

## 10. Distancia y geografía: ¿la conexión depende de la distancia?

Si los mercados están conectados por carretera, la correlación debería caer con la distancia a
Lima. La figura pone en el eje X los kilómetros y en el Y la correlación promedio con Lima.

- **Costa** en azul, **sierra** en naranja, **selva** en verde.
- Cada punto es un departamento.

![](../outputs/informe/figuras/08_integracion_distancia.png)

### Detalle elemento por elemento: figura 08

**Qué muestra.** La relación entre la distancia a Lima y la integración con Lima.

**Ejes.**
- **Eje X:** distancia en kilómetros entre la capital de el departamento y Lima (distancia geográfica
  en línea recta, no distancia por carretera).
- **Eje Y:** correlación promedio con Lima, calculada sobre los seis productos núcleo y corregida
  con la transformación de Fisher.

**Los elementos.**
- **Cada punto:** un departamento.
- **Azul:** departamento de costa.
- **Naranja:** departamento de sierra.
- **Verde:** departamento de selva.
- **Etiquetas:** nombres de los departamentos clave (Ica, Huánuco, Arequipa, Puno, Piura, Tumbes,
  Loreto, Cusco).

**Cómo se lee.**
- La nube baja de izquierda a derecha: a mayor distancia, menor correlación.
- La dispersión vertical indica que la distancia no lo explica todo.
- **Arequipa** está lejos pero tiene correlación alta: es un mercado grande y conectado.
- **Piura** y **Tumbes** son costa pero están lejos y desconectadas.
- **Loreto** está muy lejos y casi no se correlaciona.

**Qué conclusión saca.** La distancia importa, pero la etiqueta costa/sierra/selva no basta.

**Cómo se lee.** Hay una tendencia: a más distancia, menor correlación. Pero hay excepciones:
Arequipa está lejos y está bastante integrada; Piura y Tumbes son costa y están desconectadas. La
etiqueta “costa/sierra/selva” no basta: importa la conectividad real.

In [10]:
display(pd.read_csv(TAB / "tabla_13_geografia.csv").round(3))

,region,r_lima,distancia_km,zona
0,Ica,0.710,265.676,Costa
1,Huanuco,0.698,251.386,Sierra
2,La Libertad,0.663,489.267,Costa
3,Arequipa,0.579,765.665,Costa
4,Lambayeque,0.521,662.548,Costa
5,Pasco,0.464,174.461,Sierra
6,Ayacucho,0.460,329.959,Sierra
7,Ancash,0.457,285.276,Sierra
8,Tacna,0.411,985.030,Costa
9,Junin,0.362,199.008,Sierra


### Cómo se lee la tabla de geografía

- **region:** departamento.
- **r_lima:** correlación promedio con Lima (Fisher z sobre seis productos).
- **distancia_km:** distancia geográfica a Lima.
- **zona:** costa, sierra o selva.

Se ordena de mayor a menor integración. Sirve para identificar los departamentos más y menos conectadas.

**La segunda tabla** (`tabla_12_correlacion_lima.csv`) tiene una fila por
producto y departamento, con estas columnas:

- **producto:** producto.
- **region:** departamento.
- **n:** meses en común con Lima.
- **correlacion:** correlación de cambios mensuales con Lima.

Sirve para ver qué combinaciones producto-departamento están más conectadas; por ejemplo,
limón en Huánuco (0,95) o huevos en Ica (0,93).


## 11. Estacionariedad: qué es y por qué importa

Una serie es **estacionaria** si su nivel medio y su dispersión no cambian sistemáticamente con el
tiempo. Es como una persona que mantiene su peso: puede fluctuar, pero no crece sin parar.

Cuando se relacionan niveles que no son estacionarios, una regresión puede parecer convincente solo
porque ambas series tienen tendencia. Por eso el estudio contrasta estacionariedad antes de
interpretar el término de ajuste del ECM; como los resultados son mixtos, no supone cointegración
para todos los productos.

Se usan dos pruebas:

- **ADF (Dickey–Fuller aumentada):** la hipótesis nula es “hay raíz unitaria” (no estacionaria).
  Si p < 0,05, se rechaza esa hipótesis.
- **KPSS:** la hipótesis nula es “es estacionaria”. Si p < 0,05, se rechaza esa hipótesis.

Las dos pruebas se leen al revés. Por eso se usan juntas.

In [11]:
d = panel[panel.categoria == "Cebolla"].pivot_table(
    index="fecha", columns="region", values="precio_kg")[["Puno", "Lima"]].loc["2012":"2023"]
for col in ["Puno", "Lima"]:
    serie = np.log(d[col].dropna())
    p = adfuller(serie, autolag="AIC")[1]
    print(f"ADF log-{col}: p = {p:.4f} -> {'estacionaria' if p < 0.05 else 'no estacionaria'}")

ADF log-Puno: p = 0.0000 -> estacionaria
ADF log-Lima: p = 0.0000 -> estacionaria


**Cómo se lee.** Para cebolla, ADF y KPSS respaldan I(0). En arroz, ADF no rechaza raíz unitaria
en niveles y la primera diferencia sí resulta estacionaria, lo que respalda I(1). Aceite en Puno
queda **mixto**: ADF en primera diferencia da p = 0,231 y no rechaza raíz unitaria. En consecuencia,
el estudio interpreta el ECM en niveles con cautela y no presupone cointegración universal.

In [12]:
display(pd.read_csv(TAB / "tabla_04_estacionariedad.csv").round(4))

,producto,serie,adf_c,adf_ct,kpss_c,adf_diff,decision
0,Papa,Puno,0.0005,0.0015,0.1000,0.0000,I(0)
1,Papa,Lima,0.0005,0.0119,0.0404,0.0000,mixto
2,Cebolla,Puno,0.0000,0.0000,0.1000,0.0000,I(0)
3,Cebolla,Lima,0.0000,0.0000,0.1000,0.0000,I(0)
4,Zanahoria,Puno,0.0100,0.0000,0.0100,0.0000,mixto
5,Zanahoria,Lima,0.0020,0.0005,0.0935,0.0000,I(0)
6,Arroz,Puno,0.3871,0.0198,0.0100,0.0000,I(1)
7,Arroz,Lima,0.6431,0.5043,0.0100,0.0000,I(1)
8,Aceite,Puno,0.0983,0.1650,0.0499,0.2314,mixto
9,Aceite,Lima,0.9911,0.9646,0.0100,0.0000,I(1)


### Cómo se lee la tabla de estacionariedad

- **producto / serie:** qué producto y si es Puno o Lima.
- **adf_c:** p-valor de ADF con constante. p < 0,05 → se rechaza raíz unitaria.
- **adf_ct:** p-valor de ADF con constante y tendencia; se usa como especificación de robustez.
- **kpss_c:** p-valor de KPSS. p < 0,05 → se rechaza estacionariedad.
- **adf_diff:** p-valor de ADF sobre la primera diferencia.
- **decision:** I(0) cuando ADF en nivel rechaza y KPSS no; I(1) cuando las pruebas en nivel sugieren
  raíz unitaria y ADF de la primera diferencia la rechaza; mixto si las pruebas discrepan o la
  primera diferencia tampoco es estacionaria. Por eso Aceite–Puno es mixto (p = 0,231).

La tabla de resumen cuenta cuántas departamentos caen en cada categoría. La tabla de rezagos muestra
q_aic, q_bic, q_elegido, AIC, BIC, Ljung–Box y R².

![](../outputs/informe/figuras/21_adf_region.png)

### Detalle elemento por elemento: figura 21

**Qué muestra.** El p-valor de la prueba ADF (raíz unitaria) para cada producto y departamento.

**Ejes.**
- **Filas:** productos.
- **Columnas:** departamentos.
- Cada celda es el p-valor de ADF sobre el log-precio de esa serie.

**Los elementos.**
- **Verde:** p-valor bajo (< 0,05). Se rechaza la raíz unitaria: la serie se comporta como
  **estacionaria**.
- **Rojo:** p-valor alto. No se rechaza la raíz unitaria: la serie se comporta como **no
  estacionaria**.
- **Blanco:** no hay datos suficientes para esa combinación.

**Cómo se lee.**
- Una fila mayoritariamente verde indica un producto estacionario en casi todas los departamentos.
- Una fila roja indica un producto con tendencia o raíz unitaria.
- La mezcla de colores indica que el orden de integración depende de el departamento: no se puede
  generalizar.

**Advertencia.** El color es un p-valor, no la magnitud del efecto. Un verde no significa “más
grande”, sino “más evidencia contra la raíz unitaria”.

**Cómo se lee.** Verde = p-valor bajo (se rechaza raíz unitaria, estacionaria); rojo = p-valor alto
(no estacionaria). No hay un patrón único: depende del producto y de el departamento.

## 12. Rezagos: mirar meses anteriores

Un **rezago** es el valor de la variable en un mes anterior. `Δprecio_Lima(t−1)` es el cambio de
Lima el mes pasado. Si el precio tarda en llegar a Puno, el rezago ayuda a explicarlo.

¿Cuántos rezagos usar? Se prueba q = 0, 1, 2, 3 y 4, y se elige el que minimiza el **BIC**
(Criterio de Información Bayesiano). El BIC premia el ajuste pero castiga la complejidad: si un
rezago adicional no mejora lo suficiente, no se usa.

**Analogía.** Es como elegir cuántas maletas llevar: cada una puede ser útil, pero también pesa.
El BIC decide cuántas valen la pena.

In [13]:
display(pd.read_csv(TAB / "tabla_05_rezagos.csv").round(4))

,producto,q_aic,q_bic,q_elegido,aic,bic,lb_p,r2
0,Papa,1,0,0,-165.8837,-157.1679,0.1028,0.3427
1,Cebolla,0,0,0,-108.1257,-99.6168,0.2429,0.5180
2,Zanahoria,0,0,0,-85.8145,-77.6059,0.0375,0.1799
3,Arroz,0,0,0,-400.0392,-391.5304,0.0005,0.1246
4,Aceite,0,0,0,-480.7158,-472.1132,0.0119,0.1592
5,Huevos,0,0,0,-353.5173,-344.7139,0.8180,0.1998


### Cómo se lee la tabla de rezagos

- **q_aic / q_bic:** número de rezagos que prefiere cada criterio (0 a 4).
- **q_elegido:** el que se usa en el modelo; en este estudio, el de BIC.
- **aic / bic:** valores del criterio en el q elegido. Menor es mejor.
- **lb_p:** p-valor de Ljung–Box sobre los residuos. p < 0,05 indica autocorrelación que el modelo
  no capturó.
- **r2:** proporción de variación explicada.

Si q_aic y q_bic difieren, se reporta el de BIC y se prueba el otro como sensibilidad.

![](../outputs/informe/figuras/22_aic_bic.png)

### Detalle elemento por elemento: figura 22

**Qué muestra.** Cómo cambian el AIC y el BIC al aumentar el número de rezagos del ECM.

**Ejes.**
- **Eje X:** número de rezagos q, de 0 a 4.
- **Eje Y:** valor del criterio. **Menor es mejor.**

**Los elementos.**
- **Línea azul con círculos:** AIC (Criterio de Información de Akaike).
- **Línea naranja con cuadrados:** BIC (Criterio de Información Bayesiano).
- **Punto más bajo de cada línea:** el q preferido por ese criterio.

**Cómo se lee.**
- Ambas curvas suelen tener forma de U: al principio bajan porque los rezagos explican más; luego
  suben porque el castigo por complejidad pesa más.
- Si el mínimo está en q = 0, el modelo no necesita rezagos: el cambio de Lima del mismo mes basta.
- El BIC castiga más la complejidad que el AIC; por eso suele elegir un q menor o igual.

**Qué conclusión saca.** En los seis productos, el BIC elige q = 0. Ese es el modelo que se reporta.

**Cómo se lee.** Las curvas caen y luego suben. El mínimo indica el número de rezagos preferido.
En los seis productos, el mínimo está en q = 0: el cambio de Lima del mismo mes es suficiente; los
rezagos adicionales no aportan.

## 13. El modelo central: ECM

ECM significa **Error Correction Model** (modelo de corrección de error). En este estudio se estima
una relación con dos partes:

1. **Corto plazo:** cuánto del cambio de Lima se asocia con el cambio regional del mismo mes.
2. **Ajuste del desvío:** si el precio regional quedó separado del de Lima el mes anterior, `γ`
   indica si el cambio siguiente tiende a reducir esa diferencia.

La ecuación, en palabras:

> El cambio de precio en el departamento este mes depende del cambio de Lima este mes (`β₀`) y de qué tan
> lejos quedó el precio regional del de Lima el mes pasado (`γ`). Si `γ` es negativo, el cambio
> estimado corrige el desvío.

- **β₀** es el **traspaso inmediato**. Como el modelo usa logaritmos, `β₀ = 0,5` significa que un
  cambio de Lima de aproximadamente 10 % se asocia con uno de 5 % en el departamento, para cambios no muy
  grandes.
- **γ** es la velocidad de ajuste. Debe estar entre −2 y 0 para que la fórmula de vida media usada
  aquí tenga sentido.
- **Vida media** = `ln(0,5) / ln(1+γ)`: cuántos meses tarda en corregirse la mitad de un desvío.
  Si `γ = −0,28`, la vida media es ~2,1 meses.

**Precaución.** Las pruebas de estacionariedad son mixtas y Engle–Granger se presenta como evidencia
complementaria; no se afirma que todos los productos tengan una relación de largo plazo demostrada.

**Analogía.** El precio regional es un resorte atado al precio de Lima. `β₀` mide cuánto cambia de
inmediato; `γ` mide si el desvío tiende a corregirse.

In [14]:
import sys
sys.path.insert(0, str(RAIZ / "scripts"))
from ecm import construir, estimar, vida_media

d = panel[panel.categoria == "Cebolla"].pivot_table(
    index="fecha", columns="region", values="precio_kg")[["Puno", "Lima"]]
# La función compartida repone el calendario mensual antes de construir diferencias y rezagos.
datos = construir(np.log(d["Puno"]), np.log(d["Lima"]), q=0).loc["2012-01-01":"2023-12-01"]
modelo = estimar(datos, hac=True)
beta0 = modelo.params["dl0"]
gamma = modelo.params["ect"]
vida = vida_media(gamma)
print(f"N = {int(modelo.nobs)} observaciones mensuales válidas")
print(f"β₀ (traspaso inmediato) = {beta0:.3f}")
print(f"γ  (velocidad de ajuste) = {gamma:.3f}")
print(f"Vida media = {vida:.2f} meses")
print(f"R² = {modelo.rsquared:.3f}")

N = 126 observaciones mensuales válidas
β₀ (traspaso inmediato) = 0.509
γ  (velocidad de ajuste) = -0.277
Vida media = 2.14 meses
R² = 0.518


**Cómo se lee.** Para cebolla, β₀ ≈ 0,51: algo más de la mitad del cambio de Lima llega el mismo
mes. γ ≈ −0,28: el desvío se corrige; la mitad en ~2,1 meses. R² ≈ 0,52: el modelo explica la mitad
de la variación mensual.

La tabla completa de Puno:

In [15]:
display(pd.read_csv(TAB / "tabla_06_ecm_puno.csv").round(3))

,producto,q,n,beta0,beta0_p,gamma,gamma_p,vida_media,vida_ic_inf,vida_ic_sup,r2,p_coint
0,Papa,0,135,0.307,0.000,-0.306,0.000,1.897,1.350,2.795,0.343,0.000
1,Cebolla,0,126,0.509,0.000,-0.277,0.000,2.138,1.552,3.526,0.518,0.000
2,Zanahoria,0,114,0.177,0.017,-0.219,0.000,2.809,1.745,6.108,0.180,0.001
3,Arroz,0,126,0.123,0.171,-0.164,0.045,3.868,1.425,34.222,0.125,0.156
4,Aceite,0,130,0.237,0.013,-0.100,0.000,6.609,4.139,16.348,0.159,0.234
5,Huevos,0,139,0.144,0.018,-0.179,0.000,3.507,2.087,9.501,0.200,0.006


### Cómo se lee la tabla del ECM

- **producto:** producto analizado.
- **q:** número de rezagos usados.
- **n:** observaciones usadas en la regresión.
- **beta0:** traspaso inmediato. 0,5 significa que una diferencia logarítmica de Lima de 1 % se
  asocia, aproximadamente, con 0,5 % en Puno ese mes.
- **beta0_p:** p-valor de beta0. p < 0,05 = estadísticamente distinto de cero.
- **gamma:** velocidad de ajuste. Debe ser negativo; más negativo = ajuste más rápido.
- **gamma_p:** p-valor de gamma.
- **vida_media:** meses para corregir la mitad de un desvío.
- **vida_ic_inf / vida_ic_sup:** intervalo de confianza bootstrap al 95 %.
- **r2:** proporción de variación explicada.
- **p_coint:** p-valor de Engle–Granger; p < 0,05 sugiere equilibrio de largo plazo.

La tabla por producto resume mediana de beta0, gamma y vida media entre departamentos, y cuántos pares
son significativos. La tabla de diagnóstico muestra Ljung–Box, Breusch–Pagan y Jarque–Bera. La de
asimetría compara el traspaso cuando Lima sube y cuando baja.

**Cómo se lee.** Cada fila es un producto. β₀ alto indica un traspaso inmediato mayor; una vida
media baja indica un ajuste más rápido. La cebolla y la papa se ajustan en ~2 meses; el aceite tarda
~6,6. Es un patrón descriptivo compatible con diferencias entre productos; por sí solo no demuestra
que frescura, contratos o inventarios sean la causa.

La tabla por departamento permite ver si Puno es típico o atípico:

In [16]:
display(pd.read_csv(TAB / "tabla_07_ecm_producto.csv").round(3))

,producto,pares,n_mediana,beta0_mediana,gamma_mediana,vida_mediana,r2_mediana,pares_significativos
0,Cebolla,22,138.0,0.565,-0.301,1.938,0.516,21
1,Papa,20,135.5,0.335,-0.283,2.086,0.341,20
2,Zanahoria,23,125.0,0.249,-0.222,2.767,0.295,23
3,Huevos,22,140.0,0.298,-0.219,2.810,0.275,22
4,Aceite,21,127.0,0.279,-0.165,3.851,0.195,12
5,Arroz,22,137.5,0.168,-0.084,7.932,0.127,17


### Cómo se lee el resumen por producto

- **pares:** cuántas departamentos tienen estimación válida.
- **n_mediana:** meses usados, mediana entre departamentos.
- **beta0_mediana:** traspaso inmediato típico.
- **gamma_mediana:** velocidad de ajuste típica.
- **vida_mediana:** vida media típica, en meses.
- **r2_mediana:** ajuste típico.
- **pares_significativos:** cuántas departamentos tienen gamma significativo.

Sirve para saber si el resultado de Puno es representativo o excepcional.

![](../outputs/informe/figuras/09_vida_media_puno.png)

### Detalle elemento por elemento: figura 09

**Qué muestra.** La vida media del desvío entre Puno y Lima, por producto.

**Panel izquierdo (barras).**
- **Eje Y:** producto.
- **Eje X:** vida media en meses.
- **Barra azul:** vida media estimada. Barra corta = ajuste rápido; barra larga = ajuste lento.
- **Línea negra horizontal sobre la barra:** intervalo de confianza del 95 %, obtenido por
  bootstrap. Indica el rango plausible del valor.
- **Si el intervalo es muy ancho:** la estimación es imprecisa (por ejemplo, arroz).

**Panel derecho (cajas).**
- **Eje Y:** producto.
- **Eje X:** vida media en meses.
- **Caja:** rango intercuartílico (p25 a p75) de las estimaciones entre departamentos.
- **Línea naranja dentro de la caja:** mediana entre departamentos.
- **Bigotes:** valores que no son outliers (hasta 1,5 veces el rango intercuartílico).
- **Puntos sueltos:** departamentos atípicas.

**Cómo se lee.**
- Cebolla y papa tienen barras cortas e intervalos estrechos: ajuste rápido y bien estimado.
- Aceite tiene barra larga: ajuste lento.
- Arroz tiene un intervalo enorme: no se puede precisar su vida media.
- La caja del panel derecho muestra si todas los departamentos se comportan igual o si hay mucha
  dispersión.

**Nota.** El eje X está recortado en 12 meses para que las barras sean legibles. El intervalo del
arroz llega a 51,4 meses y queda fuera de escala; el texto al pie lo indica.

**Cómo se lee.** Barra = vida media; línea horizontal = intervalo de confianza bootstrap. Si el
intervalo es ancho, la estimación es imprecisa. El arroz tiene un intervalo enorme: su vida media
no se puede precisar bien.

![](../outputs/informe/figuras/10_vida_media_mapa.png)

### Detalle elemento por elemento: figura 10

**Qué muestra.** La vida media por producto y departamento, como mapa de calor.

**Ejes.**
- **Filas:** productos.
- **Columnas:** departamentos.
- Cada celda es la vida media estimada para ese producto en ese departamento, en meses.

**Los elementos.**
- **Amarillo:** vida media corta (ajuste rápido).
- **Naranja:** intermedia.
- **Rojo:** vida media larga (ajuste lento).
- **Blanco:** no hay datos suficientes para estimar.
- **Color recortado en 12 meses:** los valores mayores se pintan igual que 12, pero la tabla tiene
  el valor exacto.

**Cómo se lee.**
- Una fila amarilla indica un producto que se ajusta rápido en casi todas los departamentos.
- Una fila roja indica un producto lento.
- Una columna roja indica un departamento lenta para casi todos los productos.
- La heterogeneidad se ve como un tablero de colores, no como un patrón único.

**Cómo se lee.** Cada fila es un producto y cada columna un departamento. Amarillo = ajuste rápido;
rojo = lento; blanco = sin datos suficientes. La heterogeneidad es real: no hay un solo “mercado
peruano”.

### 13.1 Diagnóstico: ¿el modelo deja residuos ordenados?

Un **residuo** es lo que el modelo no explica: valor observado menos valor estimado. Si los
residuos tienen autocorrelación (se parecen entre meses) o heterocedasticidad (su dispersión
cambia), el modelo está mal especificado.

- **Ljung–Box:** prueba autocorrelación. p < 0,05 = residuos correlacionados.
- **Breusch–Pagan:** prueba heterocedasticidad. p < 0,05 = dispersión no constante.

El arroz falla ambas: por eso se reporta con cautela.

In [17]:
display(pd.read_csv(TAB / "tabla_18_diagnostico.csv").round(3))

,producto,n,ljung_box_4,ljung_box_8,breusch_pagan,jarque_bera,se_gamma_ols,se_gamma_hac,r2
0,Papa,135,0.103,0.107,0.354,0.000,0.046,0.059,0.343
1,Cebolla,126,0.243,0.469,0.228,0.359,0.048,0.043,0.518
2,Zanahoria,114,0.038,0.169,0.230,0.334,0.047,0.058,0.180
3,Arroz,126,0.001,0.004,0.000,0.000,0.041,0.082,0.125
4,Aceite,130,0.012,0.002,0.017,0.000,0.025,0.026,0.159
5,Huevos,139,0.818,0.903,0.954,0.000,0.033,0.043,0.200


### Cómo se lee el diagnóstico del ECM

- **producto:** producto.
- **n:** observaciones.
- **ljung_box_4 / ljung_box_8:** p-valores de autocorrelación a 4 y 8 rezagos. p < 0,05 = residuos
  autocorrelacionados.
- **breusch_pagan:** p-valor de heterocedasticidad. p < 0,05 = dispersión no constante.
- **jarque_bera:** p-valor de normalidad de residuos. p < 0,05 = no normales.
- **se_gamma_ols / se_gamma_hac:** error estándar de gamma con OLS y con HAC. Si HAC es mucho mayor,
  la inferencia OLS era optimista.
- **r2:** ajuste.

El arroz falla Ljung–Box y Breusch–Pagan; por eso se reporta con cautela.

![](../outputs/informe/figuras/19_residuos_ecm.png)

### Detalle elemento por elemento: figura 19

**Qué muestra.** Los residuos del ECM para cada producto.

**Ejes.**
- **Eje X:** número de observación, en orden temporal.
- **Eje Y:** residuo = valor observado − valor estimado por el modelo. Puede ser positivo o
  negativo.

**Los elementos.**
- **Puntos azules:** cada residuo.
- **Línea negra horizontal en 0:** referencia. Si el modelo está bien, los puntos deben fluctuar
  alrededor de esa línea.

**Cómo se lee.**
- Nube aleatoria alrededor de 0 → buen ajuste.
- Forma de embudo (más dispersión en un extremo) → heterocedasticidad.
- Ondas o rachas → autocorrelación.
- Valores muy alejados → outliers que el modelo no captura.

**Qué conclusión saca.** El arroz muestra patrones; por eso se reporta con cautela y se usan
errores HAC.

![](../outputs/informe/figuras/20_acf_ecm.png)

### Detalle elemento por elemento: figura 20

**Qué muestra.** La autocorrelación de los residuos del ECM, mes a mes.

**Ejes.**
- **Eje X:** rezago en meses (1 a 12).
- **Eje Y:** autocorrelación, entre −1 y 1.

**Los elementos.**
- **Barras azules:** autocorrelación muestral en cada rezago.
- **Banda sombreada:** intervalo de confianza aproximado del 95 % bajo la hipótesis de que no hay
  autocorrelación.

**Cómo se lee.**
- Si todas las barras caen dentro de la banda, los residuos se comportan como ruido: no queda
  estructura por explicar.
- Si alguna barra sale de la banda, el modelo deja autocorrelación: falta algo (un rezago, una
  variable o una corrección).
- El arroz tiene barras fuera de la banda; los demás están dentro.

**Qué conclusión saca.** El ECM funciona razonablemente para cinco productos; para arroz está mal
especificado o la relación es más débil.

**Cómo se lee.** En los residuos no debería haber patrones: idealmente son ruido alrededor de cero.
En el ACF, las barras no deberían salir de las bandas. El arroz muestra barras que se salen.

## 14. Asimetría: ¿sube rápido y baja lento?

La **asimetría del traspaso** pregunta si un aumento de Lima llega más rápido que una baja. Es un
fenómeno conocido como *rockets and feathers*: los precios suben como cohete y bajan como pluma.

Se estima separando el cambio de Lima en positivo y negativo. Si los coeficientes son distintos,
hay asimetría. En este estudio los indicios no alcanzan significancia al 5 %: se reporta como
exploración, no como conclusión.

In [18]:
display(pd.read_csv(TAB / "tabla_11_asimetria.csv").round(3))

,producto,n,beta_sube,beta_baja,diferencia,p_igualdad
0,Papa,135,0.489,0.097,0.392,0.097
1,Cebolla,126,0.484,0.531,-0.047,0.756
2,Zanahoria,114,0.354,-0.007,0.361,0.110
3,Arroz,126,0.012,0.285,-0.273,0.522
4,Aceite,130,0.328,-0.010,0.337,0.424
5,Huevos,139,0.237,-0.013,0.250,0.238


### Cómo se lee la tabla de asimetría

- **producto:** producto.
- **n:** observaciones.
- **beta_sube:** traspaso cuando Lima sube.
- **beta_baja:** traspaso cuando Lima baja.
- **diferencia:** beta_sube − beta_baja. Positiva = las subidas llegan más que las bajadas.
- **p_igualdad:** p-valor de la prueba de igualdad. p < 0,05 = asimetría significativa.

Ningún producto alcanza significancia al 5 %; por eso se reporta como exploración.

## 15. Predicción: nowcast y pronóstico

- **Nowcast:** estimar el cambio del mes corriente usando información de ese mismo mes (por ejemplo,
  el cambio de Lima ya observado). No es operativo si Lima todavía no publicó sus datos.
- **Pronóstico:** estimar el precio del mes siguiente, cuando aún no se conoce el precio de Lima de
  ese mes.

El baseline del nowcast es **predecir cero cambio**. El del pronóstico es la **caminata aleatoria**:
“el precio de mañana será el de hoy”. Si un modelo no supera su baseline, no mejora esa referencia.

### Métricas

| Métrica | Qué mide | Mejor si… |
|---|---|---|
| **R²** | proporción de variación explicada | más alto (1 es perfecto) |
| **MAE** | error absoluto promedio, en unidades | más bajo |
| **Mejora del MAE (%)** | reducción del MAE frente a la caminata aleatoria: `100 × (1 − MAE_modelo / MAE_RW)` | positiva; negativa significa que empeora |
| **Acierto direccional** | % de meses con el signo correcto, entre meses con cambio distinto de cero | más alto que 50 % |
| **Diebold–Mariano** | si dos modelos difieren en precisión | p < 0,05 |

**Cuidado con R².** En niveles de precio puede salir alto porque el precio de hoy se parece al de
ayer. Por eso el nowcast se evalúa sobre **cambios**, donde R² permite ver si se captura variación
mensual por encima de una referencia simple.

In [19]:
d = panel[panel.categoria == "Cebolla"].pivot_table(
    index="fecha", columns="region", values="precio_kg")[["Puno", "Lima"]].loc["2012":"2023"].dropna()
cambio = np.log(d["Puno"]).diff().dropna()
print(f"MAE de predecir cero cambio: {cambio.abs().mean():.4f}")
print("Ese es el umbral que un modelo debe superar.")

MAE de predecir cero cambio: 0.1528


Ese es el umbral que un modelo debe superar.


## 16. El torneo de modelos

Se comparan cuatro modelos:

1. **OLS (regresión lineal):** ajusta una recta; interpretable.
2. **Ridge:** como OLS, pero penaliza coeficientes grandes; útil cuando las variables están
   correlacionadas.
3. **Random Forest:** muchos árboles que promedian sus predicciones; captura relaciones no lineales.
4. **LightGBM:** boosting; árboles que se corrigen sucesivamente y que pueden ser útiles con datos
   grandes.

**Ventanas y muestra.** Desarrollo evalúa enero de 2018–diciembre de 2020 (36 meses calendario);
los modelos se entrenan con observaciones previas a 2018. Para entrar en el ranking, cada producto
necesita al menos 60 observaciones de entrenamiento y 20 de evaluación. Papa, arroz, aceite y huevos
cumplen; cebolla tiene 17 observaciones de evaluación y zanahoria 52 de entrenamiento y 17 de
evaluación, por eso ambas quedan fuera del ranking de desarrollo. El holdout sí incluye los seis
productos, de enero de 2021 a octubre de 2026. El panel llega a noviembre, pero ese mes queda fuera por el corte configurado en octubre: no se
verificó su fecha de cierre/publicación. No se afirma que noviembre esté incompleto o erróneo.

**¿Qué modelo queda primero?** Random Forest obtiene el mejor rango promedio de R² entre esos
cuatro productos elegibles (1,50), apenas por delante de Ridge (1,75). Es el resultado de ese
criterio y de esos datos; no demuestra que Random Forest sea mejor para todos los productos.

In [20]:
display(pd.read_csv(TAB / "tabla_08_torneo.csv").round(3))
display(Markdown((RAIZ / "outputs" / "analisis" / "06_modelos" / "torneo_ganador.md").read_text(encoding="utf-8")))

,modelo,rank_promedio,r2_medio,r2_mediano,acierto_medio,productos_rankeados
0,Random Forest,1.50,-0.687,-0.097,46.879,4
1,Ridge,1.75,-0.661,-0.137,46.228,4
2,OLS,2.75,-0.689,-0.153,46.228,4
3,LightGBM,4.00,-2.434,-0.544,42.785,4


# Resultado del ranking de nowcast

**Objetivo principal:** anticipacion contemporanea del cambio mensual.

- **Modelo adoptado:** Random Forest
- Rango promedio por R2 en desarrollo: 1.50
- R2 mediano en desarrollo: -0.097
- Ranking calculado sobre 4 productos con muestras suficientes: Papa, Arroz, Aceite, Huevos
- Excluidos del ranking por no alcanzar los mínimos muestrales: Cebolla, Zanahoria
- Mínimos por producto para rankear: 60 observaciones de entrenamiento y 20 de evaluación
- Holdout: 3 de 6 productos con R2 > 0; 3 con acierto > 55 %
- Compuerta: superada

### Holdout por producto

| Producto | R2 | Acierto | n |
|---|---:|---:|---:|
| Papa | 0.270 | 69.5 % | 60 |
| Cebolla | 0.218 | 64.4 % | 60 |
| Zanahoria | 0.080 | 52.9 % | 52 |
| Arroz | -0.842 | 41.9 % | 31 |
| Aceite | -0.103 | 50.0 % | 24 |
| Huevos | -0.013 | 55.3 % | 38 |

## Justificacion

La ventana de desarrollo es 2018-2020 (36 meses calendario) y se usa para evaluar/rankear; cada modelo se entrena con las observaciones disponibles antes de 2018. Solo se rankean productos con al menos 60 observaciones de entrenamiento y 20 de evaluación. El R2 es la metrica principal del nowcast porque el MAE sobre cambios favorece predecir cero. El holdout 2021-01 a 2026-10 se uso una sola vez; noviembre queda fuera porque no se verifico su fecha de cierre/publicacion.

Por que no gana LightGBM si tiene mayor acierto en el holdout? LightGBM acierta el signo en 62.2 % de los meses, frente a 58.0 % de Random Forest, pero el acierto es una lectura binaria: trata igual un error de 0,1 % que uno de 5 %. El ganador se eligio en desarrollo con R2, donde Random Forest tiene rango promedio 1.50 y acierto medio 46.9 %, contra 4.00 y 42.8 % de LightGBM. En el holdout Random Forest tiene menor MAE que LightGBM en 5 de 6 productos y R2 positivo en papa, cebolla y zanahoria; LightGBM tiene R2 muy negativo en arroz y zanahoria. Cambiar el ganador segun el holdout seria seleccionar sobre el conjunto de prueba. La matriz de confusion se reporta como diagnostico complementario, no como criterio de seleccion.

## Pronostico a un mes: holdout

Cambio porcentual del MAE frente a la caminata aleatoria: positivo = mejora; negativo = empeora.

| Producto | N | OLS | Ridge | Random Forest | LightGBM |
|---|---:|---:|---:|---:|---:|
| Papa | 62 | +14.5 % | +12.7 % | -74.2 % | -85.6 % |
| Cebolla | 62 | +8.3 % | +9.9 % | -2.8 % | -12.5 % |
| Zanahoria | 53 | -4.7 % | -7.7 % | -62.0 % | -59.8 % |
| Arroz | 36 | +3.9 % | +7.7 % | -282.0 % | -353.2 % |
| Aceite | 24 | -37.8 % | -36.6 % | -303.1 % | -306.8 % |
| Huevos | 38 | -3.6 % | -3.3 % | -146.4 % | -142.3 % |

Ridge y OLS mejoran en Papa, Cebolla, Arroz; ambos empeoran en Zanahoria, Aceite, Huevos. Random Forest y LightGBM empeoran frente al baseline en los seis productos. La referencia de Lima aporta valor contemporaneo, pero no ofrece un pronostico general a un mes.


### Cómo se lee el ranking del torneo

- **modelo:** OLS, Ridge, Random Forest o LightGBM.
- **rank_promedio:** puesto promedio por R² en el desarrollo 2018–2020, entre los cuatro productos
  que cumplen los mínimos de muestra.
- **r2_medio / r2_mediano:** R² promedio y mediano de esos cuatro productos.
- **acierto_medio:** porcentaje de signos correctos en la ventana de evaluación, sin contar los meses con cambio exactamente cero.

Cebolla y zanahoria no entran en el promedio de desarrollo por datos válidos insuficientes; sí
aparecen en el holdout. No hay que leer la Tabla 15 como un ranking equilibrado de los seis productos.

![](../outputs/informe/figuras/11_ranking_torneo.png)

### Detalle elemento por elemento: figura 11

**Qué muestra.** El ranking de los cuatro modelos del torneo.

**Ejes.**
- **Eje Y:** modelo (OLS, Ridge, Random Forest, LightGBM).
- **Eje X:** rango promedio por R² en desarrollo 2018–2020. Menor es mejor.
- **Barra:** longitud = rango promedio.

**Cómo se lee.** Random Forest tiene el rango promedio más bajo (1,50); Ridge queda cerca (1,75).
Ese promedio se calcula sobre **cuatro productos elegibles**: papa, arroz, aceite y huevos. Cebolla
y zanahoria no alcanzan los mínimos muestrales de desarrollo.

**Advertencia.** El R² mediano del modelo primero es negativo y el holdout es desigual. Ser primero en
este criterio no garantiza alta precisión ni que gane en todos los productos.

![](../outputs/informe/figuras/12_ganador_vs_observado.png)

### Detalle elemento por elemento: figura 12

**Qué muestra.** El cambio mensual observado y las predicciones de los cuatro modelos en los seis
productos del holdout.

**Ejes.**
- **Eje X:** tiempo, enero de 2021–octubre de 2026.
- **Eje Y:** cambio logarítmico mensual, mostrado en porcentaje aproximado (`100 × Δlog(P)`).

**Los elementos.**
- **Línea negra:** valor observado (lo que pasó).
- **Líneas de color:** predicciones de OLS, Ridge, Random Forest y LightGBM.
- **Línea gris horizontal:** cero. Por encima = subió; por debajo = bajó.

**Cómo se lee.** Si una línea sigue la dirección de la negra, acierta el signo; si exagera o suaviza
los picos, hay error de magnitud. El gráfico ya no muestra solo al ganador: permite comparar los
cuatro modelos en los seis productos. Random Forest tiene R² 0,270 en papa, 0,218 en cebolla y 0,080
en zanahoria; en arroz, aceite y huevos el R² es negativo.

![](../outputs/informe/figuras/13_curvas_aprendizaje.png)

### Detalle elemento por elemento: figura 13

**Qué muestra.** Cómo cambia el error de cada modelo al aumentar el tamaño del entrenamiento en
una curva de cebolla.

**Ejes.**
- **Eje X:** observaciones mensuales válidas de entrenamiento: 33, 46, 56 o 67. Son datos
  anteriores a 2018, no meses tomados de la ventana de desarrollo 2018–2020.
- **Eje Y (panel izquierdo):** MAE (menor es mejor).
- **Eje Y (panel derecho):** R² (mayor es mejor).

Cada punto se evalúa sobre las **17 observaciones válidas de cebolla** dentro de 2018–2020. La
ventana tiene 36 meses calendario, pero por faltantes no hay 36 filas utilizables para este producto.
En esta curva, Random Forest reduce MAE y aumenta R² con más entrenamiento; LightGBM mejora en el
último tramo. Es un resultado de esta curva, no una ley general.

**Cómo se lee el ranking.** Random Forest queda primero por rango promedio (1,50) y Ridge segundo
(1,75). **Cómo se lee la figura de predicciones.** Las líneas comparan los cambios mensuales reales
y las predicciones de los cuatro modelos en los seis productos del holdout; no es una curva
exclusiva del ganador. **Cómo se leen las curvas.** Para cebolla, la curva de aprendizaje muestra la
mejora de Random Forest con más meses; no debe generalizarse ese patrón a todos los productos.

## 17. Cómo se ve un árbol y qué aprende

Un **árbol de decisión** hace preguntas en cadena: “¿el cambio de Lima fue mayor que 0,21?”. Si sí,
va a una rama; si no, a otra. Cada hoja termina en una predicción.

El primer árbol del Random Forest se parte por `dl0` (cambio de Lima), luego por `dp2` (cambio
propio de hace dos meses) y por `sin1` (estacionalidad). Es la misma señal que el ECM.

El **árbol sustituto** de LightGBM es un árbol simple entrenado para imitar las predicciones del
ensemble; sirve para ver qué variables domina sin dibujar cientos de árboles.

La **importancia** mide cuánto usa el modelo cada variable. La **dependencia parcial** muestra cómo
cambia la predicción cuando una variable se mueve, manteniendo el resto fijo.

![](../outputs/informe/figuras/15_arbol_rf.png)

### Detalle elemento por elemento: figura 15

**Qué muestra.** El primer árbol del Random Forest, recortado a profundidad 3.

**Cada caja (nodo).**
- **Primera línea:** la pregunta o condición. Por ejemplo, `dl0 <= 0.215` significa “¿el cambio de
  Lima fue menor o igual a 0,215?”.
- **`samples`:** porcentaje de observaciones que llegan a ese nodo.
- **`value`:** predicción promedio del cambio de precio en ese nodo.
- **Color:** intensidad de la predicción (más naranja = valor más alto).

**Las flechas.**
- **True:** camino si la condición se cumple.
- **False:** camino si no se cumple.

**Las hojas.**
- Nodos sin flechas: predicción final.
- **`(...)`:** el árbol continúa, pero se recortó para que se vea.

**Cómo se lee.**
- La primera división (raíz) es la variable más importante: `dl0`, el cambio de Lima.
- Luego aparecen `dp2` (cambio propio de hace dos meses) y `sin1` (estacionalidad).
- Siguiendo un camino se obtiene una regla: “si Lima subió poco y el precio propio venía bajando,
  la predicción es X”.

![](../outputs/informe/figuras/16_arbol_lgbm.png)

### Detalle elemento por elemento: figura 16

**Qué muestra.** Un **árbol sustituto** de LightGBM.

**Qué es un árbol sustituto.** LightGBM es un conjunto de cientos de árboles; no se puede dibujar
uno solo que lo represente. El sustituto es un árbol simple entrenado para imitar las predicciones
del ensemble. No es el modelo original, pero muestra qué variables y cortes usa.

**Los elementos son los mismos que en la figura 15.**
- Caja: condición, `samples`, `value`, color.
- Flechas: True/False.
- Hojas: predicción.

**Cómo se lee.**
- Si el sustituto se parte por `dl0` y `dp2`, el ensemble está usando la misma señal que el ECM.
- Si se partiera por variables sin sentido, sería una señal de sobreajuste.

![](../outputs/informe/figuras/17_importancia_rf.png)

### Detalle elemento por elemento: figura 17

**Qué muestra.** La importancia de cada variable en el Random Forest.

**Ejes.**
- **Eje Y:** variable.
- **Eje X:** importancia. Es un número entre 0 y 1; la suma de todas las importancias es 1.

**Los elementos.**
- **Barras horizontales:** una por variable.
- **Barra más larga:** variable más usada por el modelo para dividir.

**Cómo se lee.**
- `dl0` (cambio de Lima) arriba: el modelo usa la referencia de Lima.
- `dp1`, `dp2` (cambios propios): persistencia.
- `sin1`, `cos1`: estacionalidad.
- Una variable con barra casi nula aporta poco.

**Advertencia.** Importancia no es causalidad. Una variable puede ser importante porque está
correlacionada con otra, no porque la cause.

![](../outputs/informe/figuras/18_pdp.png)

### Detalle elemento por elemento: figura 18

**Qué muestra.** La **dependencia parcial** de tres variables en el Random Forest.

**Qué es.** Para cada valor de una variable, se mueve esa variable manteniendo las demás fijas y se
promedia la predicción. Así se ve el efecto aislado de esa variable.

**Ejes.**
- **Eje X:** valor de la variable (`dl0`, `dp1` o `cos1`).
- **Eje Y:** predicción promedio del cambio de precio.

**Los elementos.**
- **Línea:** la relación promedio.
- **Si la línea sube:** al aumentar la variable, la predicción sube.
- **Si es plana:** la variable casi no afecta.

**Cómo se lee.**
- Una línea con pendiente en `dl0` indica que el cambio de Lima sí mueve la predicción.
- Una línea plana indica que, controlando el resto, esa variable no aporta.

**Cómo se lee.** En el árbol, cada caja es una pregunta; el color, la predicción promedio; el
ancho, la cantidad de datos. En importancia, barras más largas = variables más usadas. En PDP, la
curva muestra el efecto de una variable sobre la predicción.

## 18. Robustez: ¿el resultado aguanta cambios?

Un resultado es robusto si no depende de una decisión arbitraria. Se prueba:

- **q = 0, 1, 2:** distintos números de rezagos.
- **Sin 2021–2023:** se excluye el shock inflacionario posterior a la pandemia.
- **Ventana 2008–2023:** se usa más historia donde existe.
- **Papa amarilla:** variedad fuente adicional, fuera de las nueve categorías del panel; solo seis
  meses de cruce con Lima, por eso no se estima el ECM.

Los cambios de N también importan: q2 reduce dos observaciones en zanahoria (114 a 112), mientras
que la ventana desde 2008 agrega 16 observaciones en arroz (126 a 142) y 38 en aceite (130 a 168).
Si una conclusión cambia —como la significancia del arroz— hay que decirlo.

In [21]:
robustez = pd.read_csv(TAB / "tabla_10_robustez.csv")
robustez["escenario"] = robustez["escenario"].replace({
    "sin_shock": "sin shock", "ventana_2008": "ventana 2008",
    "papa_amarilla": "papa amarilla"})
display(robustez.round(3))

,producto,escenario,q,n,beta0,gamma,vida_media,r2,p_gamma
0,Papa,base,0,135,0.307,-0.306,1.897,0.343,0.000
1,Papa,q0,0,135,0.307,-0.306,1.897,0.343,0.000
2,Papa,q1,1,135,0.261,-0.254,2.368,0.362,0.000
3,Papa,q2,2,135,0.265,-0.238,2.552,0.366,0.000
4,Papa,sin shock,0,99,0.295,-0.327,1.747,0.360,0.000
5,Papa,ventana 2008,0,135,0.307,-0.306,1.897,0.343,0.000
6,Cebolla,base,0,126,0.509,-0.277,2.138,0.518,0.000
7,Cebolla,q0,0,126,0.509,-0.277,2.138,0.518,0.000
8,Cebolla,q1,1,126,0.497,-0.263,2.268,0.519,0.000
9,Cebolla,q2,2,126,0.497,-0.267,2.236,0.519,0.000


### Cómo se lee la tabla de robustez

- **producto:** producto.
- **escenario:** base, q0, q1, q2, sin_shock, ventana_2008 o papa_amarilla.
- **q:** rezagos usados en ese escenario.
- **n:** observaciones.
- **beta0:** traspaso inmediato.
- **gamma:** velocidad de ajuste.
- **vida_media:** meses.
- **r2:** ajuste.
- **p_gamma:** significancia.

Se compara la fila base con las demás. Si los valores cambian poco, el resultado es robusto; si
cambian mucho, se reporta la sensibilidad.

![](../outputs/informe/figuras/14_robustez.png)

### Detalle elemento por elemento: figura 14

**Qué muestra.** La vida media del ECM en distintos escenarios de robustez.

**Ejes.**
- **Filas:** productos.
- **Columnas:** escenarios:
  - `base`: q elegido por BIC.
  - `q0`, `q1`, `q2`: distintos números de rezagos.
  - `sin_shock`: excluye 2021–2023.
  - `ventana_2008`: usa más historia.
  - `papa_amarilla`: variedad alternativa.
- **Cada celda:** vida media en meses; el número está escrito dentro.

**Los elementos.**
- **Amarillo:** vida media corta.
- **Rojo:** vida media larga.
- **Blanco:** no estimable (por ejemplo, papa amarilla tiene solo 6 meses de cruce).

**Cómo se lee.**
- Una fila con valores parecidos entre columnas es robusta.
- Una fila que cambia mucho indica que la conclusión depende del escenario.
- El arroz pierde significancia sin el shock; el aceite duplica su vida media.

**Cómo se lee.** Cada fila es un producto y cada columna un escenario. El arroz pierde
significancia con q2 (p = 0,1234) y sin el periodo 2021–2023 (p = 0,0822); al extender la ventana a
2008 queda apenas significativo (p = 0,0461). El aceite continúa significativo al excluir el shock,
pero su vida media aumenta de 6,6 a 10,1 meses. La conclusión, por tanto, no es idéntica en todos
los escenarios.

## 19. Veredicto, en lenguaje llano

1. **Los mercados están conectados, pero no son uno solo.** Lima e Ica se mueven bastante juntas;
   Lima y Tumbes, mucho menos.
2. **El traspaso es incompleto.** Si Lima sube 10 %, el efecto inmediato estimado en Puno varía por
   producto; `β₀` aproxima el porcentaje transmitido cuando los cambios son pequeños.
3. **La velocidad depende del producto.** Cebolla y papa se ajustan en ~2 meses; aceite tarda ~6,6.
4. **Hay valor predictivo contemporáneo**, pero el ranking de desarrollo se calcula solo en cuatro
   productos elegibles. Random Forest queda primero; el holdout de seis productos muestra R²
   positivo en papa, cebolla y zanahoria.
5. **El pronóstico a un mes es heterogéneo.** Ridge y OLS mejoran frente a la caminata aleatoria en
   papa, cebolla y arroz; ambos empeoran en zanahoria, aceite y huevos. Random Forest y LightGBM
   empeoran frente a la caminata aleatoria en los seis productos, como muestra la tabla comparativa.
6. **La geografía se asocia con integración, sobre todo por distancia**, pero la etiqueta
   costa/sierra/selva no explica por sí sola las diferencias.

En la especificación base del ECM, γ es significativo en los seis productos; el arroz pierde
significancia con q2 y al excluir 2021–2023, de modo que ese hallazgo no es robusto a todos los
escenarios.

## 20. Qué no se puede afirmar

- No se puede decir que Lima **cause** el precio de Puno: hay relación, no experimento.
- No se puede decir que el clima no importa: no se incluyó clima en este estudio.
- No se puede decir que el modelo sirve para cualquier año: el shock 2021–2023 cambia algunas
  estimaciones.
- No se puede decir que es un nowcast operativo: no se verificó el rezago de publicación.
- No se puede decir que Random Forest es el mejor para todo producto o situación: solo queda
  primero en el ranking de desarrollo de los cuatro productos con muestra suficiente.